### <span style="color:Skyblue"> Predict the Customer Churn  </span>

In [ ]:
from tensorflow.keras.models import load_model
import pickle
import pandas as pd

# Load the trained model
model = load_model('trained_model.h5')

# Load scaler and Encoders
def load_pickle_file(filepath):
    try:
        with open(filepath, 'rb') as file:
            return pickle.load(file)
    except FileNotFoundError:
        print(f"Error: File not found - {filepath}")
        return None

scaler = load_pickle_file("dataScaler.pkl")
gender_encoder = load_pickle_file("gender_encoder.pkl")
geo_encoder = load_pickle_file("geography_encoder.pkl")

# Example input data
input_data = {
    'CreditScore': 800,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 55,
    'Tenure': 3,
    'Balance': 50000,
    'NumOfProducts': 1,
    'HasCrCard': 1,
    'IsActiveMember': 0,
    'EstimatedSalary': 30000
}

# Function to preprocess input data
def preprocess_input(data):
    try:
        # Validate input keys
        expected_keys = [
            'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 
            'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'
        ]

        missing_keys = [key for key in expected_keys if key not in data]
        if missing_keys:
            raise ValueError(f"Missing keys in input data: {missing_keys}")

        # Convert to DataFrame
        input_dataset = pd.DataFrame([data])

        # One-hot encode Geography
        geo_encoded = geo_encoder.transform([[input_data['Geography']]]).toarray()
        geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_encoder.get_feature_names_out(['Geography']))
        
        # Label encode Gender
        input_dataset['Gender'] = gender_encoder.fit_transform(input_dataset['Gender'])

        # Concatenate one-hot encoded Geography
        input_dataset = pd.concat([input_dataset.drop('Geography', axis=1), geo_encoded_df], axis=1)

        # Scale numerical features
        scaled_data = scaler.transform(input_dataset)

        return scaled_data
    
    except Exception as e:
        print(f"Error during preprocessing: {e}")
        return None

# Preprocess the input data
input_scaled = preprocess_input(input_data)
if input_scaled is None:
    raise ValueError("Error: Preprocessing failed. Check the input data and preprocessing steps.")


# Predict churn probability
prediction = model.predict(input_scaled)
prediction_proba = prediction[0][0]

# Output result
print(f"Churn Probability: {prediction_proba:.2f}")
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")


### <span style="color:lightgreen"> Predict the Customer Churn | Multiple Data  </span>

In [26]:
from tensorflow.keras.models import load_model
import pickle
import pandas as pd

# Load the trained model
model = load_model('trained_model.h5')

# Load scaler and Encoders
def load_pickle_file(filepath):
    try:
        with open(filepath, 'rb') as file:
            return pickle.load(file)
    except FileNotFoundError:
        print(f"Error: File not found - {filepath}")
        return None

scaler = load_pickle_file("dataScaler.pkl")
gender_encoder = load_pickle_file("gender_encoder.pkl")
geo_encoder = load_pickle_file("geography_encoder.pkl")

# Function to preprocess input data
def preprocess_input(data):
    try:
        # Validate input keys
        expected_keys = [
            'CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 
            'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'
        ]

        missing_keys = [key for key in expected_keys if key not in data]
        if missing_keys:
            raise ValueError(f"Missing keys in input data: {missing_keys}")

        # Convert to DataFrame
        input_dataset = pd.DataFrame([data])

        # One-hot encode Geography
        geo_encoded = geo_encoder.transform([[input_data['Geography']]]).toarray()
        geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_encoder.get_feature_names_out(['Geography']))
        
        # Label encode Gender
        input_dataset['Gender'] = gender_encoder.fit_transform(input_dataset['Gender'])

        # Concatenate one-hot encoded Geography
        input_dataset = pd.concat([input_dataset.drop('Geography', axis=1), geo_encoded_df], axis=1)

        # Scale numerical features
        scaled_data = scaler.transform(input_dataset)

        return scaled_data
    
    except Exception as e:
        print(f"Error during preprocessing: {e}")
        return None

# Example input data
# Input data as a list of dictionaries
input_data_list = [
    {
        'CreditScore': 750, 'Geography': 'Spain', 'Gender': 'Female', 
        'Age': 35, 'Tenure': 5, 'Balance': 80000, 
        'NumOfProducts': 2, 'HasCrCard': 1, 
        'IsActiveMember': 0, 'EstimatedSalary': 95000
    },
    {
        'CreditScore': 620, 'Geography': 'Germany', 'Gender': 'Male', 
        'Age': 45, 'Tenure': 8, 'Balance': 120000, 
        'NumOfProducts': 1, 'HasCrCard': 0, 
        'IsActiveMember': 1, 'EstimatedSalary': 70000
    },
    {
        'CreditScore': 800, 'Geography': 'France', 'Gender': 'Female', 
        'Age': 29, 'Tenure': 3, 'Balance': 60000, 
        'NumOfProducts': 2, 'HasCrCard': 1, 
        'IsActiveMember': 1, 'EstimatedSalary': 85000
    },
    {
        'CreditScore': 510, 'Geography': 'Spain', 'Gender': 'Female', 
        'Age': 38, 'Tenure': 4, 'Balance': 0, 
        'NumOfProducts': 1, 'HasCrCard': 1, 
        'IsActiveMember': 0, 'EstimatedSalary': 118913
    },
    {
        'CreditScore': 669, 'Geography': 'France', 'Gender': 'Male', 
        'Age': 46, 'Tenure': 3, 'Balance': 0, 
        'NumOfProducts': 2, 'HasCrCard': 0, 
        'IsActiveMember': 1, 'EstimatedSalary': 8487
    }
]

# Process each customer and predict churn
for i, input_data in enumerate(input_data_list):
    # Preprocess input data
    input_scaled = preprocess_input(input_data)
    if input_scaled is None:
        print(f"Error: Preprocessing failed for customer {i + 1}. Skipping this customer.")
        continue

    # Predict churn probability
    prediction = model.predict(input_scaled)
    prediction_proba = prediction[0][0]

    # Output result for each customer
    print(f"\nCustomer {i + 1} - Churn Prediction")
    print(f"Input Data: {input_data}")
    print(f"Churn Probability: {prediction_proba:.2f}")
    if prediction_proba > 0.5:
        print("The customer is likely to churn.")
    else:
        print("The customer is not likely to churn.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step

Customer 1 - Churn Prediction
Input Data: {'CreditScore': 750, 'Geography': 'Spain', 'Gender': 'Female', 'Age': 35, 'Tenure': 5, 'Balance': 80000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 0, 'EstimatedSalary': 95000}
Churn Probability: 0.12
The customer is not likely to churn.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

Customer 2 - Churn Prediction
Input Data: {'CreditScore': 620, 'Geography': 'Germany', 'Gender': 'Male', 'Age': 45, 'Tenure': 8, 'Balance': 120000, 'NumOfProducts': 1, 'HasCrCard': 0, 'IsActiveMember': 1, 'EstimatedSalary': 70000}
Churn Probability: 0.52
The customer is likely to churn.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step

Customer 3 - Churn Prediction
Input Data: {'CreditScore': 800, 'Geography': 'France', 'Gender': 'Female', 'Age': 29, 'Tenure': 3, 'Balance': 60000, 'NumOfProducts': 2, 'HasCrCard': 1, 'IsActiveMember': 1, 'EstimatedSalary': 85000}
Churn Probability: 0.04
The customer is not likely to churn.
1/1 ━━━━━━━━

/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(
